# POC: CAVLS

📖 対応記事: `親子対話：CAVLS`🔗 [記事を読む](https://github.com/bonsai/sound-gen/blob/main/articles/%E8%A6%AA%E5%AD%90%E5%AF%BE%E8%A9%B1%EF%BC%9ACAVLS.md)

In [ ]:
# @title Setup
!pip install -q torch numpy matplotlib

In [ ]:
# @title Demo
# CAVLS: Content Adaptive Variable Length Segmentation
import numpy as np
import matplotlib.pyplot as plt

# Simulate adaptive vs fixed segmentation
sr = 16000
dur = 3.0
t = np.linspace(0, dur, int(sr*dur))

# Audio with varying complexity
audio = np.zeros_like(t)
audio[int(sr*0.0):int(sr*0.8)] = 0.05 * np.random.randn(int(sr*0.8))  # silence/noise
audio[int(sr*0.8):int(sr*2.2)] = np.sin(2*np.pi*300*t[int(sr*0.8):int(sr*2.2)]) * 0.8  # speech
audio[int(sr*2.2):] = np.sin(2*np.pi*600*t[int(sr*2.2):]) * 0.3 + 0.1*np.random.randn(int(sr*0.8))  # complex

# Fixed segmentation (50ms blocks)
fixed_len = int(sr * 0.05)
fixed_segments = []
for i in range(0, len(audio), fixed_len):
  fixed_segments.append(np.std(audio[i:i+fixed_len]))

# Adaptive: larger segments for silence, smaller for complex
adaptive_segments = []
idx = 0
while idx < len(audio):
  complexity = np.std(audio[idx:idx+int(sr*0.05)])
  seg_len = int(sr * (0.1 if complexity < 0.05 else 0.025))
  seg_len = min(seg_len, len(audio) - idx)
  adaptive_segments.append(complexity)
  idx += seg_len

fixed_times = np.arange(len(fixed_segments)) * fixed_len / sr
adaptive_times = np.linspace(0, dur, len(adaptive_segments))

plt.figure(figsize=(12, 5))
plt.subplot(311); plt.plot(t, audio); plt.title('Audio Signal')
plt.ylabel('Amplitude')
plt.subplot(312); plt.stem(fixed_times[::10], fixed_segments[::10]); plt.title(f'Fixed Segmentation ({len(fixed_segments)} segments)')
plt.ylabel('Energy')
plt.subplot(313); plt.stem(adaptive_times, adaptive_segments); plt.title(f'CAVLS: Adaptive ({len(adaptive_segments)} segments)')
plt.xlabel('Time (s)')
plt.ylabel('Energy')
plt.tight_layout()
plt.show()

print(f"✅ CAVLS: Fixed={len(fixed_segments)} segments, Adaptive={len(adaptive_segments)}")
print(f"   Saves {abs(len(adaptive_segments)-len(fixed_segments))} segments ({100*abs(len(adaptive_segments)-len(fixed_segments))/len(fixed_segments):.0f}%)")

---
### ✅ CAVLS (Content Adaptive Variable Length Segmentation)

✔ 固定セグメントと適応的セグメントを比較
✔ 音声の複雑さに応じてフレーム長を変更

[▶ Colabで実行](https://colab.research.google.com/github/bonsai/sound-gen/blob/main/colabs/poc_cavls.ipynb)